In [1]:
import pandas as pd

import numpy as np

import pickle

import mlflow.sklearn

from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

In [2]:
test_file = 'F:\\Guvi Projects\\Smart_Premium\\playground-series-s4e12\\test.csv'
test_data = pd.read_csv(test_file)

In [3]:
test_data.drop(['id','Policy Start Date'], axis=1, inplace=True)

In [4]:
for col in test_data.select_dtypes(include=['int64', 'float64']).columns:
    test_data[col].fillna(test_data[col].mean(), inplace=True)

for col in test_data.select_dtypes(include='object').columns:
    test_data[col].fillna(test_data[col].mode()[0])

In [5]:
def age_category(data):
    if 18 < data <= 30: return '18-30'
    elif 30 < data <= 40: return '31-40'
    elif 40 < data <= 50: return '41-50'
    elif 50 < data <= 64: return '51-64'
    else: return '<64'

In [6]:
def dependent_category(data):
    if data == 0: return '0'
    elif 0 < data <= 2: return '0-2'
    elif 2 < data <= 3: return '2-3'
    else: return '<3'

In [7]:
def health_category(data):
    if 0 < data <= 15: return '0-15'
    elif 15 < data <= 25: return '15-25'
    elif 25 < data <= 35: return '15-35'
    else: return '<35'

In [8]:
def claims(data):
    if 0 < data <= 1: return '0-1'
    elif 1 < data <= 2: return '1-2'
    else: return '<2'

In [9]:
def vehicle(data):
    if 0 < data <= 5: return '0-5'
    elif 5 < data <= 10: return '5-10'
    elif 10 < data <= 20: return '10-20'
    else: return '<20'

In [10]:
def credit(data):
    if 0 < data <= 300: return '0-300'
    elif 300 < data <= 600: return '300-600'
    elif 600 < data < 800: return '600-800'
    else: return '<800'

In [11]:
def insurance(data):
    if 0 < data <= 3: return '0-3'
    elif 3 < data <= 6: return '3-6'
    elif 6 < data < 9: return '6-9'
    else: return '<9'

In [12]:
test_data['Age_Group'] = test_data['Age'].apply(age_category)
test_data['Dependent_Group'] = test_data['Number of Dependents'].apply(dependent_category)
test_data['Health_Group'] = test_data['Health Score'].apply(health_category)
test_data['Prev_Claims_Group'] = test_data['Previous Claims'].apply(claims)
test_data['Vehicle_Group'] = test_data['Vehicle Age'].apply(vehicle)
test_data['Credit_Group'] = test_data['Credit Score'].apply(credit)
test_data['Insurance_Group'] = test_data['Insurance Duration'].apply(insurance)

In [13]:
mappings = {
    "Education Level": {"High School": 0, "Bachelor's": 1, "Master's": 2, "PhD": 3},
    "Customer Feedback": {"Poor": 0, "Average": 1, "Good": 2},
    "Exercise Frequency": {"Rarely": 0, "Weekly": 1, "Monthly": 2, "Daily": 3},
    "Policy Type": {"Basic": 0, "Comprehensive": 1, "Premium": 2}
}

In [14]:
test_data = test_data.replace(mappings).infer_objects(copy=False)

In [15]:
columns_to_encode = test_data[['Age_Group', 'Dependent_Group', 'Health_Group', 'Prev_Claims_Group',
                               'Vehicle_Group', 'Credit_Group', 'Insurance_Group', 'Gender', 'Marital Status',
                               'Occupation', 'Location', 'Smoking Status', 'Property Type']]

In [16]:
le = LabelEncoder()
for col in columns_to_encode.columns:
    test_data[col] = le.fit_transform(test_data[col])

In [17]:
encoded_test_data = pd.DataFrame({
    'Age': test_data['Age_Group'],
    'Gender': test_data['Gender'],
    'Annual Income': test_data['Annual Income'],
    'Marital Status': test_data['Marital Status'],
    'Number of Dependents': test_data['Dependent_Group'],
    'Education Level': test_data['Education Level'],
    'Occupation': test_data['Occupation'],
    'Health Score': test_data['Health_Group'],
    'Location': test_data['Location'],
    'Policy Type': test_data['Policy Type'],
    'Previous Claims': test_data['Prev_Claims_Group'],
    'Vehicle Age': test_data['Vehicle_Group'],
    'Credit Score': test_data['Credit_Group'],
    'Insurance Duration': test_data['Insurance_Group'],
    'Customer Feedback': test_data['Customer Feedback'],
    'Smoking Status': test_data['Smoking Status'],
    'Exercise Frequency': test_data['Exercise Frequency'],
    'Property Type': test_data['Property Type']
})

In [18]:
def log_transform(data, columns_to_transform):
    for col in columns_to_transform:
        data[f'{col}_log'] = np.log1p(data[col])  
        data[f'{col}_msle'] = (data[f'{col}_log'] - data[f'{col}_log'].mean())**2  
        data[col] = np.sqrt(data[f'{col}_msle'])  
        data.drop(columns=[f'{col}_log', f'{col}_msle'], inplace=True)  
    
    return data

In [19]:
log_transform(encoded_test_data, ['Annual Income'])

,Age,Gender,Annual Income,Marital Status,Number of Dependents,Education Level,Occupation,Health Score,Location,Policy Type,Previous Claims,Vehicle Age,Credit Score,Insurance Duration,Customer Feedback,Smoking Status,Exercise Frequency,Property Type
0,0,0,2.047339,3,3,1,1,0,0,0,1,1,1,0,0.0,1,1,2
1,1,0,1.951517,1,1,2,1,0,1,2,1,1,1,2,2.0,1,0,0
2,2,0,0.046350,0,0,3,2,1,2,1,1,1,3,3,1.0,1,2,1
3,0,0,0.530245,0,2,3,1,0,1,1,0,0,2,1,0.0,1,3,2
4,0,1,0.499565,0,1,0,2,0,1,2,1,1,2,2,1.0,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
799995,2,0,0.772963,1,1,1,3,0,0,2,1,2,1,0,1.0,1,3,1
799996,2,0,1.411763,2,0,2,3,0,0,0,1,3,1,0,2.0,0,3,0
799997,0,0,0.675430,2,0,2,0,0,2,1,1,2,1,1,0.0,0,2,0
799998,1,0,0.936247,2,2,2,3,1,2,2,1,1,1,2,1.0,0,1,1


In [20]:
with open("best_model.pkl", "rb") as file:
    model = pickle.load(file)

In [21]:
predictions = model.predict(encoded_test_data)

In [22]:
test_data['Predicted_Premium_Amount'] = predictions

In [23]:
test_data.to_csv("F:\\Guvi Projects\\Smart_Premium\\research_data\\Test_Predictions.csv", index=False)

In [24]:
pickle_path = 'F:\\Guvi Projects\\Smart_Premium\\pickles\\test_predictions.pkl'

with open(pickle_path, "wb") as file:
    pickle.dump(predictions, file)

print('test_predictions.pkl saved successfully...')

test_predictions.pkl saved successfully...


In [25]:
mlflow.set_tracking_uri("http://localhost:5000")

with mlflow.start_run():
    mlflow.sklearn.log_model(model, "Insurance_Premium_Model")
    mlflow.log_params({"Model": "Best Model from Training"})
    mlflow.log_artifact("Test_Predictions.csv")
    
    for i in range(5):
        mlflow.log_metric(f"Predicted Premium {i}", predictions[i])

print("Predictions logged in MLflow and saved to Test_Predictions.csv.")

2025/02/23 14:58:37 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



MlflowException: API request to http://localhost:5000/api/2.0/mlflow/runs/create failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/create (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x00000274364DB5E0>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))